# 0 – InterGATE: Main Pipeline

Entrenamiento completo del modelo GNN para clasificación de subtipos moleculares de cáncer de mama.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from intergate.config import CFG
from intergate.utils import set_seed, cleanup_memory, get_device
from intergate.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only, make_dataloaders,
    build_regulator_features,
)
from intergate.graph import build_backbone
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.losses import FocalLoss, compute_metrics_full, compute_class_weights_balanced
from intergate.training import predict_proba_xh_mode, train_graph_learning, finetune_pruned
from intergate.pruning import export_pruned_graph, evaluate_keep_ratios
from intergate.stability import edge_set_from_edge_index, pairwise_jaccard_stats
from intergate.ablation import (
    AblationConfig, build_model_from_cfg, make_prior_for_cfg,
    run_single_seed, run_ablation, register_runtime,
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


## 1) Configuración

Modifica aquí los hiperparámetros si es necesario.

In [ ]:
# ── Override defaults if needed ──
# CFG.DATA_DIR ya apunta por defecto a PROJECT_ROOT/data/processed.
# CFG.LR = 1e-3
# CFG.EPOCHS1 = 90
print("Config OK")
print("DATA_DIR:", CFG.DATA_DIR)
print("EXPR_CSV:", CFG.EXPR_CSV)
print("META_CSV:", CFG.META_CSV)

## 2) Cargar datos + genes

In [ ]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)
print("Classes:", classes)

## 3) Grafo backbone (HuRI + OmniPath) con caché

Usa `get_or_build_backbone` (graph_cache) que envuelve `build_backbone` (graph.py).
Si existe un caché validado se carga directamente; si no, se reconstruye
desde HuRI.psi + OmniPath TSV, se deduplicican aristas y se guarda.

La validación comprueba:
- **REQUIRED_CONNECTED_GENES** (FAM118A, TMEM30A, TMEM30B)
- **EXPECTED_CONNECTED_GENES** (referencia `genes_usados.csv` si existe)


In [ ]:
# ── Opción A: con doble capa de caché (graph_cache → build_backbone) ──
# get_or_build_backbone valida + cachea en pipeline_cache/
# y build_backbone a su vez cachea en backbone_cache/.
#
# Opción B (directa): build_backbone ya tiene su propio caché.
# Descomenta la línea B y comenta la A si prefieres un solo nivel.

# --- A) Doble caché (recomendada, replica el flujo del notebook Grafo) ---
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=CFG.FORCE_REBUILD_HURI_CACHE,
    use_omnipath=CFG.USE_OMNIPATH,
    use_huri=CFG.USE_HURI,
)

# --- B) Caché directo (solo backbone_cache/) ---
# edge_index, edge_weight, edge_type = build_backbone(
#     genes_kegg,
#     use_omnipath=CFG.USE_OMNIPATH,
#     use_huri=CFG.USE_HURI,
#     force_rebuild=CFG.FORCE_REBUILD_HURI_CACHE,
# )

print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


## 4) Split + escalado + connected-only

In [ ]:
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )

## 5) Features de reguladores (Xs_graph) con caché

Usa `get_or_build_Xh` (graph_cache) o directamente `build_regulator_features`.
Las features dependen de Xs_gene (escalado), genes conectados y edge_index,
así que un cambio en split/semilla/escalado invalida automáticamente el caché.


In [ ]:
# --- A) Con caché (recomendada, replica el flujo del notebook Grafo) ---
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False,
    stats=CFG.REG_STATS,
    min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)

# --- B) Sin caché (directo) ---
# Xs_graph, graph_feat_names = build_regulator_features(
#     Xs_gene, genes_kegg, edge_index,
#     add_features=CFG.ADD_REGULATOR_FEATURES,
#     stats=CFG.REG_STATS,
#     min_genes=CFG.REG_MIN_GENES,
#     max_regulators=CFG.REG_MAX_REGULATORS,
# )

print(f"Xs_graph: {Xs_graph.shape} | features: {len(graph_feat_names)}")


## 6) DataLoaders

In [ ]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx,
    batch_size=CFG.BATCH_SIZE,
)

## 7) Tensores de grafo

In [ ]:
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

# Adjacency sparse (para sparse_mm_batch)
N = Xs_gene.shape[1]
adj = torch.sparse_coo_tensor(
    edge_index_t, edge_weight_t, size=(N, N),
).coalesce().to(DEVICE)
print("adj:", adj.shape)

## 8) Ablación + entrenamiento

In [ ]:
register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg,
    X_h=Xs_graph, y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t,
    edge_type_t=edge_type_t,
    reg_groups_t=reg_groups_t if 'reg_groups_t' in dir() else None,
)

In [ ]:
# Configuración de ablación (solo FULL por defecto)
CFG_FULL = AblationConfig(
    name="FULL",
    edge_type_gating=CFG.EDGE_TYPE_GATING,
    sample_cond_gating=CFG.SAMPLE_COND_GATING,
    sample_cond_mode=CFG.SAMPLE_COND_MODE,
    use_signed_prior=True,
    signed_channels_mode=CFG.SIGNED_CHANNELS_MODE,
    connectivity_penalty=CFG.ADD_CONNECTIVITY_PENALTY,
    do_pretrain=CFG.DO_PRETRAIN,
    do_stability=CFG.DO_STABILITY_SELECTION,
)

ABLA_CONFIGS = [CFG_FULL]
ABLA_SEEDS = [1234, 1, 42, 369]

# Ejecutar ablación
df_abla = run_ablation(ABLA_CONFIGS, ABLA_SEEDS, save_root=CFG.ARTIFACTS_ROOT)
display(df_abla)



In [ ]:
# ── Ablaciones individuales (descomenta las que quieras)
ABLA_CONFIGS = [
    CFG_FULL,
    AblationConfig("no_conn",        True,  True,  "per_type", True,  "type_only",     False, CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("typed_only",     True,  False, "per_type", True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("cond_only",      False, True,  "global",   True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("no_type_no_cond",False, False, "global",   True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("no_signed",      True,  True,  "per_type", False, "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("dual_backbone",  True,  True,  "per_type", True,  "dual_backbone", True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    # AblationConfig("no_pretrain",    True,  True,  "per_type", True,  "type_only",     True,  False,           CFG.DO_STABILITY_SELECTION),
    # AblationConfig("no_stability",   True,  True,  "per_type", True,  "type_only",     True,  CFG.DO_PRETRAIN, False),
]

ABLA_SEEDS = [1234, 1, 42, 369]

df_abla = run_ablation(ABLA_CONFIGS, ABLA_SEEDS, save_root=CFG.ARTIFACTS_ROOT)
display(df_abla)

In [ ]:
# Resumen
summary = (
    df_abla.groupby("cfg")[["macro_f1","auc_ovr_macro","acc","n_edges_final","n_nodes_compact"]]
    .mean()
    .sort_values("macro_f1", ascending=False)
)
display(summary)

## 9) Post-hoc: cargar modelo ganador y evaluar

In [ ]:
from intergate.ablation import load_bundle

bundle_dir = f"{CFG.ARTIFACTS_ROOT}/FULL/seed_1234"
bundle = load_bundle(bundle_dir)
print("Bundle cargado:", list(bundle.keys()) if isinstance(bundle, dict) else "OK")